In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
data = [
    (1,"  Ajeeth Kumar ","Male",25,50000,None),
    (2,"Priya","Female",None,45000,"Chennai"),
    (3,None,"NA",29,None,"Mumbai"),
    (4,"Ravi","Male",25,52000,"Delhi"),
    (4,"Ravi","Male",'null',52000,"Delhi"),   # Duplicate
    (5,"Arun ","Male",-5,38000," "),
    (6,"Meena","NA",31,120000,"Hyderabad"),
    (7,"Kiran","Male",150,45000,"Pune"),
    (8,"Anu","Female",22,'Missing',"Bangalore"),
    (9,"-","Male",27,70000,"Chennai"),
    (None,None,None,None,None,None)
]

In [0]:
columns=["id","name","gender","age","salary","city"]

In [0]:
df = spark.createDataFrame(data,columns)

In [0]:
df.display()

id,name,gender,age,salary,city
1,Ajeeth Kumar,Male,25,50000,null
2,Priya,Female,null,45000,Chennai
3,null,NA,29,null,Mumbai
4,Ravi,Male,25,52000,Delhi
4,Ravi,Male,null,52000,Delhi
5,Arun,Male,-5,38000,
6,Meena,NA,31,120000,Hyderabad
7,Kiran,Male,150,45000,Pune
8,Anu,Female,22,Missing,Bangalore
9,-,Male,27,70000,Chennai


In [0]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- city: string (nullable = true)



In [0]:
df.select([
    count(when(col(c).isNull(),c)).alias(c)
    for c in df.columns
]).display()

id,name,gender,age,salary,city
1,2,1,2,2,2


In [0]:
df = df.dropDuplicates(['id'])

In [0]:
df

DataFrame[id: bigint, name: string, gender: string, age: string, salary: string, city: string]

In [0]:
df = df.withColumn('name',when(col('name')=="-",None).otherwise(col('name')))

In [0]:
df.display()

id,name,gender,age,salary,city
1,Ajeeth Kumar,Male,25,50000,null
2,Priya,Female,null,45000,Chennai
3,null,NA,29,null,Mumbai
4,Ravi,Male,25,52000,Delhi
5,Arun,Male,-5,38000,
6,Meena,NA,31,120000,Hyderabad
7,Kiran,Male,150,45000,Pune
8,Anu,Female,22,Missing,Bangalore
9,null,Male,27,70000,Chennai
null,null,null,null,null,null


In [0]:
df = df.withColumn('gender',when(col('gender')=="NA",None).otherwise(col('gender')))

In [0]:
df.display()

id,name,gender,age,salary,city
1,Ajeeth Kumar,Male,25,50000,null
2,Priya,Female,null,45000,Chennai
3,null,null,29,null,Mumbai
4,Ravi,Male,25,52000,Delhi
5,Arun,Male,-5,38000,
6,Meena,null,31,120000,Hyderabad
7,Kiran,Male,150,45000,Pune
8,Anu,Female,22,Missing,Bangalore
9,null,Male,27,70000,Chennai
null,null,null,null,null,null


In [0]:
df = df.withColumn('salary',when(col('salary')=="Missing",None).otherwise(col('salary')))

In [0]:
df = df.withColumn(
    "city",
    when(trim(col("city")) == "", None)
    .otherwise(col("city"))
)

In [0]:
df.display()

id,name,gender,age,salary,city
1,Ajeeth Kumar,Male,25,50000,null
2,Priya,Female,null,45000,Chennai
3,null,null,29,null,Mumbai
4,Ravi,Male,25,52000,Delhi
5,Arun,Male,-5,38000,null
6,Meena,null,31,120000,Hyderabad
7,Kiran,Male,150,45000,Pune
8,Anu,Female,22,null,Bangalore
9,null,Male,27,70000,Chennai
null,null,null,null,null,null


In [0]:
df = df.withColumn("age",col("age").cast("int"))\
       .withColumn("salary",col("salary").cast("double"))

In [0]:
df.display()

id,name,gender,age,salary,city
1,Ajeeth Kumar,Male,25,50000.0,null
2,Priya,Female,null,45000.0,Chennai
3,null,null,29,null,Mumbai
4,Ravi,Male,25,52000.0,Delhi
5,Arun,Male,-5,38000.0,null
6,Meena,null,31,120000.0,Hyderabad
7,Kiran,Male,150,45000.0,Pune
8,Anu,Female,22,null,Bangalore
9,null,Male,27,70000.0,Chennai
null,null,null,null,null,null


In [0]:
df.na.drop().display()

id,name,gender,age,salary,city
4,Ravi,Male,25,52000.0,Delhi
7,Kiran,Male,150,45000.0,Pune


In [0]:
df.na.drop(how = 'all').display()

id,name,gender,age,salary,city
1,Ajeeth Kumar,Male,25,50000.0,null
2,Priya,Female,null,45000.0,Chennai
3,null,null,29,null,Mumbai
4,Ravi,Male,25,52000.0,Delhi
5,Arun,Male,-5,38000.0,null
6,Meena,null,31,120000.0,Hyderabad
7,Kiran,Male,150,45000.0,Pune
8,Anu,Female,22,null,Bangalore
9,null,Male,27,70000.0,Chennai


In [0]:
df.na.drop(how = 'any').display()

id,name,gender,age,salary,city
4,Ravi,Male,25,52000.0,Delhi
7,Kiran,Male,150,45000.0,Pune


In [0]:
df.display()

id,name,gender,age,salary,city
1,Ajeeth Kumar,Male,25,50000.0,null
2,Priya,Female,null,45000.0,Chennai
3,null,null,29,null,Mumbai
4,Ravi,Male,25,52000.0,Delhi
5,Arun,Male,-5,38000.0,null
6,Meena,null,31,120000.0,Hyderabad
7,Kiran,Male,150,45000.0,Pune
8,Anu,Female,22,null,Bangalore
9,null,Male,27,70000.0,Chennai
null,null,null,null,null,null


In [0]:
df.na.drop(how = 'any',subset = ['name','gender']).display()

id,name,gender,age,salary,city
1,Ajeeth Kumar,Male,25,50000.0,null
2,Priya,Female,null,45000.0,Chennai
4,Ravi,Male,25,52000.0,Delhi
5,Arun,Male,-5,38000.0,null
7,Kiran,Male,150,45000.0,Pune
8,Anu,Female,22,null,Bangalore


In [0]:
df.na.drop(thresh = 4).display()

id,name,gender,age,salary,city
1,Ajeeth Kumar,Male,25,50000.0,null
2,Priya,Female,null,45000.0,Chennai
4,Ravi,Male,25,52000.0,Delhi
5,Arun,Male,-5,38000.0,null
6,Meena,null,31,120000.0,Hyderabad
7,Kiran,Male,150,45000.0,Pune
8,Anu,Female,22,null,Bangalore
9,null,Male,27,70000.0,Chennai


In [0]:
df.na.fill(0).display()

id,name,gender,age,salary,city
1,Ajeeth Kumar,Male,25,50000.0,null
2,Priya,Female,0,45000.0,Chennai
3,null,null,29,0.0,Mumbai
4,Ravi,Male,25,52000.0,Delhi
5,Arun,Male,-5,38000.0,null
6,Meena,null,31,120000.0,Hyderabad
7,Kiran,Male,150,45000.0,Pune
8,Anu,Female,22,0.0,Bangalore
9,null,Male,27,70000.0,Chennai
0,null,null,0,0.0,null


In [0]:
df.na.fill('unknown').display()

id,name,gender,age,salary,city
1,Ajeeth Kumar,Male,25,50000.0,unknown
2,Priya,Female,null,45000.0,Chennai
3,unknown,unknown,29,null,Mumbai
4,Ravi,Male,25,52000.0,Delhi
5,Arun,Male,-5,38000.0,unknown
6,Meena,unknown,31,120000.0,Hyderabad
7,Kiran,Male,150,45000.0,Pune
8,Anu,Female,22,null,Bangalore
9,unknown,Male,27,70000.0,Chennai
null,unknown,unknown,null,null,unknown


In [0]:
df.na.fill({
    "age":25,
    "salary":50000,
    "city":"Unknown"
}).display()

id,name,gender,age,salary,city
1,Ajeeth Kumar,Male,25,50000.0,Unknown
2,Priya,Female,25,45000.0,Chennai
3,null,null,29,50000.0,Mumbai
4,Ravi,Male,25,52000.0,Delhi
5,Arun,Male,-5,38000.0,Unknown
6,Meena,null,31,120000.0,Hyderabad
7,Kiran,Male,150,45000.0,Pune
8,Anu,Female,22,50000.0,Bangalore
9,null,Male,27,70000.0,Chennai
null,null,null,25,50000.0,Unknown
